In [ ]:
%pip install xarray netcdf4 cartopy certifi

In [ ]:
import ftplib
import os

ESA_FTP_HOST = "science-pds.cryosat.esa.int"
USERNAME = "USERNAME"
PASSWORD = "PASSWORD!"

YEARS = [2010]       # years I have so far
MONTHS = list(range(8,13)) # months 1-12
for YEAR in YEARS:
    for MONTH in MONTHS:
        BASE_PATH = f"TEMPO_SWATH_GRID/{YEAR}/{MONTH:02d}/ANTARCTIC"
        LOCAL_DIR = "./cryosat_downloads"

        os.makedirs(LOCAL_DIR, exist_ok=True)

        # Connect securely
        ftps = ftplib.FTP_TLS(ESA_FTP_HOST)
        ftps.login(USERNAME, PASSWORD)
        ftps.prot_p()
        ftps.set_pasv(True)

        print("Connected.")

        # Change directory instead of nlst(full_path)
        ftps.cwd(BASE_PATH)

        files = ftps.nlst()

        if not files:
            print("No files found.")
        else:
            for fname in files:
                if fname.endswith(".nc"):
                    local_file = os.path.join(LOCAL_DIR, fname)

                    # Skip if already downloaded
                    if os.path.exists(local_file):
                        print(f"Skipping {fname} (already exists)")
                        continue

                    print(f"Downloading {fname}")
                    with open(local_file, "wb") as f:
                        ftps.retrbinary(f"RETR {fname}", f.write)

        print("Done.")
        ftps.quit()


In [ ]:
import json
import re
from pathlib import Path

import numpy as np
import pandas as pd
import xarray as xr
import matplotlib.pyplot as plt
from pyproj import Transformer

In [ ]:
# Use the first CryoSat-2 .nc file in ./cryosat_downloads
cryosat_dir = Path(".") / "cryosat_downloads"
cryosat_files = sorted(cryosat_dir.glob("*.nc"))
if not cryosat_files:
    raise FileNotFoundError(f"No .nc files found in {cryosat_dir.resolve()}")
cryosat_nc_path = cryosat_files[0]
print(f"Using: {cryosat_nc_path}")

In [ ]:
# List .nc files under the repo (pick one for cryosat_nc_path)
for p in Path(".").resolve().rglob("*.nc"):
    print(p)

In [ ]:
# Extract lake locations from the DataGrabber notebook
project_root = Path(".").resolve().parent
notebook_path = project_root / "Notebooks" / "3.2_DataGrabberLoop.ipynb"
out_dir = project_root / "Subglacial_Lakes_Data"
out_csv = out_dir / "lake_locations.csv"

pattern = re.compile(r"locationAnomaly\(([^,]+),\s*([^,]+),\s*\"([^\"]+)\"")

with notebook_path.open("r", encoding="utf-8") as f:
    nb = json.load(f)

rows = []
for cell in nb.get("cells", []):
    for line in cell.get("source", []):
        match = pattern.search(line)
        if match:
            lon = float(match.group(1).strip())
            lat = float(match.group(2).strip())
            name = match.group(3).strip()
            rows.append((name, lon, lat))

lakes = pd.DataFrame(rows, columns=["name", "lon", "lat"])
out_dir.mkdir(parents=True, exist_ok=True)
lakes.to_csv(out_csv, index=False)
lakes.head()

In [ ]:
# Load CryoSat-2 grid
ds = xr.open_dataset(cryosat_nc_path)
var = ds[list(ds.data_vars)[0]].isel(time=0)

In [ ]:
# Pre-download Natural Earth coastlines to avoid SSL issues
import io
import zipfile
import urllib.request
from pathlib import Path
import ssl
import certifi
from cartopy import config
from cartopy.io import Downloader

ne_meta = {
    "resolution": "50m",
    "category": "physical",
    "name": "coastline",
    "config": config,
}
downloader = Downloader.from_config((
    "shapefiles", "natural_earth", "50m", "physical", "coastline"
))
target_path = downloader.target_path(ne_meta)
target_dir = Path(target_path).parent

if not Path(target_path).exists():
    target_dir.mkdir(parents=True, exist_ok=True)
    url = downloader.url(ne_meta)
    ctx = ssl.create_default_context(cafile=certifi.where())
    with urllib.request.urlopen(url, context=ctx) as resp:
        with zipfile.ZipFile(io.BytesIO(resp.read())) as zf:
            zf.extractall(target_dir)

In [ ]:
# Cartopy version (preserves Antarctic shape in projection)
import os
import ssl
import certifi
import cartopy.crs as ccrs
from matplotlib.ticker import FixedLocator

# Ensure Cartopy downloads use a valid CA bundle
os.environ.setdefault("SSL_CERT_FILE", certifi.where())
os.environ.setdefault("REQUESTS_CA_BUNDLE", certifi.where())
ssl._create_default_https_context = ssl.create_default_context(cafile=certifi.where())

fig = plt.figure(figsize=(8, 6))
ax = plt.axes(projection=ccrs.SouthPolarStereo())

# Build 2D grid for pcolormesh
x2d, y2d = np.meshgrid(var["x"].values, var["y"].values, indexing="ij")

# Plot CryoSat grid in its native CRS (EPSG:3031 ~= SouthPolarStereo)
mesh = ax.pcolormesh(
    x2d,
    y2d,
    var.values,
    transform=ccrs.SouthPolarStereo(),
    cmap="viridis",
    shading="auto",
)

# Plot lakes in lon/lat
ax.scatter(
    lakes["lon"],
    lakes["lat"],
    s=10,
    c="red",
    alpha=0.8,
    label="Lakes",
    transform=ccrs.PlateCarree(),
)

# Lat/lon labels and ticks for every gridline
gl = ax.gridlines(
    crs=ccrs.PlateCarree(),
    draw_labels=True,
    linewidth=0.4,
    color="gray",
    alpha=0.6,
    linestyle="--",
)
gl.top_labels = False
gl.right_labels = False
gl.xlocator = FixedLocator(list(range(-180, 181, 30)))
gl.ylocator = FixedLocator(list(range(-90, -59, 5)))
ax.set_xlabel("Latitude")
ax.set_ylabel("Longitude")

ax.set_title("CryoSat-2 Antarctic Grid + Lake Locations (Cartopy)")
ax.legend(loc="lower left")
fig.colorbar(mesh, ax=ax, shrink=0.7)
plt.show()